<a href="https://colab.research.google.com/github/ale153023/Ale-s-repository/blob/main/Enlaces.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
enlaces =pd.read_excel("/Enlaces.xlsx")
#enlaces.head()
urls = enlaces["Link"].tolist()

In [ ]:
import requests
def validar_url(url):
    try:
        r = requests.head(url, allow_redirects=True, timeout=10)
        return r.status_code, r.url
    except Exception as e:
        return None, str(e)

In [ ]:
from numpy import savetxt
from tqdm.notebook import tqdm
from urllib.parse import urlparse

def is_homepage_redirect(original_url, final_url):
    try:
        analisis_original = urlparse(original_url)
        analisis_final = urlparse(final_url)
        #Para validar si URL es el mismo
        if analisis_original.netloc != analisis_final.netloc:
            return False
        #Verificar home page
        homepage_paths = ["", "/", "/index.html", "/default.aspx", "/home"]
        if analisis_final.path.lower() in homepage_paths:
            return True
        return False
    except Exception:
        return False #Manejo de casos donde URL falla
resultados_validacion = []
for url in tqdm(urls, desc="Validando URLs"):
    status_code, final_url = validar_url(url)
    resultados_validacion.append({'Original Link': url, 'Status Code': status_code, 'URL Final': final_url})
df_resultados = pd.DataFrame(resultados_validacion)
df_resultados['¿Vuelve a homepage?'] = df_resultados.apply(
    lambda row: is_homepage_redirect(row['Original Link'], row['URL Final']),
    axis=1
)
display(df_resultados.head())

Validando URLs:   0%|          | 0/484 [00:00<?, ?it/s]

,Original Link,Status Code,URL Final,¿Vuelve a homepage?
0,https://www.surtidor.com/mezcladoras-monomando...,200,https://www.surtidor.com/mezcladoras-monomando...,False
1,https://www.surtidor.com/mezcladora-para-frega...,200,https://www.surtidor.com/mezcladora-para-frega...,False
2,https://www.surtidor.com/mezcladora-para-lavab...,200,https://www.surtidor.com/mezcladora-para-lavab...,False
3,https://www.surtidor.com/flange-piso-negra-25-...,200,https://www.surtidor.com/flange-piso-negra-25-...,False
4,https://www.surtidor.com/brida-mueller-na-0215...,200,https://www.surtidor.com/brida-mueller-na-0215...,False


In [89]:
import requests
from bs4 import BeautifulSoup

def disponibilidad_producto(url, timeout=10):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    try:
        response = requests.get(url, allow_redirects=True, timeout=timeout, headers=headers)
        response.raise_for_status() #Para msj HTTP
        soup = BeautifulSoup(response.text, 'html.parser')
        page_text = soup.get_text().lower()
        #Palabras para indicar disponibilidad
        disponible_keys = ['in stock', 'añadir al carrito', 'Comprar','comprar',' comprar', 'add to cart', 'buy now', 'agregar al carrito', 'Disponible',' disponible']
        no_disponible_keys = ['out of stock','agotado','no disponible', 'no está disponible', 'sold out', 'este producto actualmente no se encuentra disponible', 'este producto no está disponible por el momento.']

        disponible = any(key in page_text for key in disponible_keys)
        no_disponible = any(key in page_text for key in no_disponible_keys)

        if disponible and not no_disponible:
            return 'Disponible'
        elif no_disponible and not disponible:
            return 'No disponible'
        #elif disponible and no_disponible:
            #return 'Incierto (Conflicto de palabras clave)'
        else:
            return 'Incierto (No se encontraron palabras clave)'

    except requests.exceptions.Timeout:
        return 'Fuera de tiempo'
    except requests.exceptions.RequestException as e:
        return f'Request Error: {e}'
    except Exception as e:
        return f'Parsing Error: {e}' # Changed 'None/error' to 'Parsing Error'
df_resultados['Disponibilidad del Producto'] = df_resultados.apply(
    lambda row: 'Redirección a Homepage' if row['¿Vuelve a homepage?'] else (
        disponibilidad_producto(row['URL Final']) if row['URL Final'] else 'N/A'
    ),
    axis=1
)
display(df_resultados.head())
df_resultados.to_excel("resultados.xlsx")

,Original Link,Status Code,URL Final,¿Vuelve a homepage?,Disponibilidad del Producto
0,https://www.surtidor.com/mezcladoras-monomando...,200,https://www.surtidor.com/mezcladoras-monomando...,False,Incierto (No se encontraron palabras clave)
1,https://www.surtidor.com/mezcladora-para-frega...,200,https://www.surtidor.com/mezcladora-para-frega...,False,Incierto (No se encontraron palabras clave)
2,https://www.surtidor.com/mezcladora-para-lavab...,200,https://www.surtidor.com/mezcladora-para-lavab...,False,Incierto (No se encontraron palabras clave)
3,https://www.surtidor.com/flange-piso-negra-25-...,200,https://www.surtidor.com/flange-piso-negra-25-...,False,Incierto (No se encontraron palabras clave)
4,https://www.surtidor.com/brida-mueller-na-0215...,200,https://www.surtidor.com/brida-mueller-na-0215...,False,Incierto (No se encontraron palabras clave)


In [90]:
import requests
from bs4 import BeautifulSoup

# Filtrar df_resultados para enlaces con estado 'Incierto'
surtidor_incierto_df = df_resultados[
    (df_resultados['URL Final'].str.contains('surtidor.com', na=False)) &
    (df_resultados['Disponibilidad del Producto'].str.contains('Incierto', na=False))
]

if not surtidor_incierto_df.empty:
    # Tomar la primera URL de ejemplo
    sample_url = surtidor_incierto_df.iloc[0]['URL Final']
    print(f"URL de ejemplo de surtidor.com con estado 'Incierto': {sample_url}\n")

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    try:
        response = requests.get(sample_url, allow_redirects=True, timeout=10, headers=headers)
        response.raise_for_status() # Lanza un error para códigos de estado HTTP 4xx/5xx
        soup = BeautifulSoup(response.text, 'html.parser')
        page_text = soup.get_text().lower()

        print("--- Contenido de texto de la página ---\n")
        print(page_text[:])
        print("\n--- Fin del texto de ejemplo de la página ---\n")
        print("Por favor, revisa el texto anterior y proporciona frases exactas que indiquen la disponibilidad o no disponibilidad del producto.")

    except requests.exceptions.RequestException as e:
        print(f"Error al obtener contenido para {sample_url}: {e}")
    except Exception as e:
        print(f"Error al analizar contenido para {sample_url}: {e}")
else:
    print("No se encontraron resultados 'Incierto' para enlaces de surtidor.com.")

URL de ejemplo de surtidor.com con estado 'Incierto': https://www.surtidor.com/mezcladoras-monomandos-rugo-03351813020.html

--- Contenido de texto de la página ---

 





mezcladora para lavabo rugo 24-pa cromo















             























 iniciar sesión 
expand_more






mi cuenta


pagar



iniciar sesión





 

conmutador: 55 1450 9000  |  compra ahora: 55 5015 0289  |  sucursales |  whatsapp: 55 7609 4579 en ciudad de méxico 




            estado: cdmx y área metropolitana 




















 
							menu arrow_back









































0
$0.00


0










subtotal
$0.00






envío








total
$0.00



pagar





























 comparar ()





















baño
 
add remove  









            wc
        



            lavabo
        



            regaderas
        



            tocador
        



            tina
        






            barra de seguridad
        



            jabonera
        




In [88]:
import requests
from bs4 import BeautifulSoup

# Filtrar df_resultados para enlaces con estado 'Incierto'
surtidor_incierto_df = df_resultados[
    (df_resultados['URL Final'].str.contains('ferrecsa.com', na=False)) &
    (df_resultados['Disponibilidad del Producto'].str.contains('Incierto', na=False))
]

if not surtidor_incierto_df.empty:
    # Tomar la primera URL de ejemplo
    sample_url = surtidor_incierto_df.iloc[0]['URL Final']
    print(f"URL de ejemplo de surtidor.com con estado 'Incierto': {sample_url}\n")

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    try:
        response = requests.get(sample_url, allow_redirects=True, timeout=10, headers=headers)
        response.raise_for_status() # Lanza un error para códigos de estado HTTP 4xx/5xx
        soup = BeautifulSoup(response.text, 'html.parser')
        page_text = soup.get_text().lower()

        print("--- Contenido de texto de la página ---\n")
        print(page_text[:])
        print("\n--- Fin del texto de ejemplo de la página ---\n")
        print("Por favor, revisa el texto anterior y proporciona frases exactas que indiquen la disponibilidad o no disponibilidad del producto.")

    except requests.exceptions.RequestException as e:
        print(f"Error al obtener contenido para {sample_url}: {e}")
    except Exception as e:
        print(f"Error al analizar contenido para {sample_url}: {e}")
else:
    print("No se encontraron resultados 'Incierto' para enlaces de ferrecsa.com.")

URL de ejemplo de surtidor.com con estado 'Incierto': https://tienda.ferrecsa.com.mx/Producto/bateria-li-ion-18v-4-amp-bl1840b-makita

--- Contenido de texto de la página ---





bateria li-ion 18v 4 amp bl1840b makita* - tienda ferrecsa
 















 



tienda ferrecsainicio
 
                                    categorías
                                

productos
marcas
contacto
envíos









tienda ferrecsa
¿especialidades? lo nuestro
















0







inicio



                                    categorías
                                



productos


marcas


contacto


envios


 
 


registrarse
iniciar sesión


carrito 0





menú





inicio





        categorías
      






productos


marcas


registrarse


iniciar sesión



 













compartir producto por...




copiar enlace
whatsapp
facebook



cerrar
















bateria li-ion 18v 4 amp bl1840b makita*




























bateria li-ion 18v 4 amp bl1840b makita*

detalles generales